In [1]:
import pandas as pd 

In [ ]:
pip install libpysal

In [5]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from libpysal.weights import KNN
from esda import G_Local
import folium

# Set random seed
np.random.seed(123)

# Generate 50 random points near Boston
n_points = 50
boston_center = (-71.0589, 42.3601)  # lon, lat
radius = 0.1  # degrees, approx 11 km
longitudes = np.random.uniform(boston_center[0] - radius, boston_center[0] + radius, n_points)
latitudes = np.random.uniform(boston_center[1] - radius, boston_center[1] + radius, n_points)
values = np.random.normal(size=n_points)

# Create GeoDataFrame
df = pd.DataFrame({'longitude': longitudes, 'latitude': latitudes, 'value': values})
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Project to a metric CRS for accurate distance neighbor calculations (e.g., UTM zone 19N for Boston)
# EPSG 32619 corresponds to UTM zone 19N
gdf_utm = gdf.to_crs(epsg=32619)

# Calculate 4-nearest neighbors spatial weights using libpysal
coords = np.array([(point.x, point.y) for point in gdf_utm.geometry])
knn = KNN.from_array(coords, k=4)

# Compute Getis-Ord G* statistic (local G)
g_star = G_Local(gdf_utm['value'].values, knn, transform='r')

# Add results to GeoDataFrame
gdf_utm['G_star'] = g_star.Zs
gdf_utm['significant'] = (gdf_utm['G_star'] > 1.96) | (gdf_utm['G_star'] < -1.96)

# Reproject back to WGS84 for folium mapping
gdf = gdf_utm.to_crs(epsg=4326)

# Create folium map centered at Boston
m = folium.Map(location=[boston_center[1], boston_center[0]], zoom_start=12)

# Define colors for significant and non-significant points
def color_point(significant, g_star_val):
    if significant:
        # Hotspot (significant)
        return 'red' if g_star_val > 1.96 else 'blue'  # Red = high values hotspot, Blue = low values hotspot
    else:
        return 'gray'

# Add points to the folium map
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color=color_point(row['significant'], row['G_star']),
        fill=True,
        fill_opacity=0.7,
        popup=(f"Value: {row['value']:.2f}<br>"
               f"G*: {row['G_star']:.2f}<br>"
               f"Significant: {row['significant']}")
    ).add_to(m)

# Display the map (in notebook) or save
m.save('getis_ord_folium_map.html')
m  # If in Jupyter notebook this will display the map inline

KeyboardInterrupt: 

In [ ]:
pip install esda

In [8]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from libpysal.weights import KNN
from esda import G_Local
import folium

# Set random seed
np.random.seed(123)

# Generate random points near Boston
n_points = 50
boston_center = (-71.0589, 42.3601)  # lon, lat
radius = 0.1  # degrees

lons = np.random.uniform(boston_center[0] - radius, boston_center[0] + radius, n_points)
lats = np.random.uniform(boston_center[1] - radius, boston_center[1] + radius, n_points)
values = np.random.normal(size=n_points)

# Create GeoDataFrame
geometry = [Point(lon, lat) for lon, lat in zip(lons, lats)]
gdf = gpd.GeoDataFrame(
    {'value': values}, 
    geometry=geometry, 
    crs="EPSG:4326"
)

# Project to UTM for accurate distance calculations
gdf_utm = gdf.to_crs(epsg=32619)

# Calculate spatial weights (4 nearest neighbors)
coords = np.array([(point.x, point.y) for point in gdf_utm.geometry])
knn_weights = KNN.from_array(coords, k=4)

# Compute Getis-Ord G* statistic
g_local = G_Local(gdf_utm['value'].values, knn_weights, transform='r')

# Add results back to original GeoDataFrame
gdf['G_star'] = g_local.Zs
gdf['p_value'] = g_local.p_sim
gdf['significant'] = gdf['p_value'] < 0.05

# Classify hotspots and coldspots
def get_hotspot_type(row):
    if not row['significant']:
        return 'Not Significant'
    return 'Hotspot' if row['G_star'] > 0 else 'Coldspot'

gdf['hotspot_type'] = gdf.apply(get_hotspot_type, axis=1)

# Color mapping
color_map = {
    'Hotspot': 'red',
    'Coldspot': 'blue',
    'Not Significant': 'gray'
}

# Create map
m = folium.Map(
    location=[boston_center[1], boston_center[0]], 
    zoom_start=12
)

# Add points to map
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color=color_map[row['hotspot_type']],
        fill=True,
        fill_opacity=0.7,
        popup=(f"Value: {row['value']:.2f}<br>"
               f"G*: {row['G_star']:.2f}<br>"
               f"Type: {row['hotspot_type']}")
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 140px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
<p style="margin:0"><b>Hotspot Analysis</b></p>
<p style="margin:5px"><i class="fa fa-circle" style="color:red"></i> Hotspot (High Values)</p>
<p style="margin:5px"><i class="fa fa-circle" style="color:blue"></i> Coldspot (Low Values)</p>
<p style="margin:5px"><i class="fa fa-circle" style="color:gray"></i> Not Significant</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m.save('hotspot_analysis.html')
print(f"Map saved! Hotspots: {(gdf['hotspot_type'] == 'Hotspot').sum()}, "
      f"Coldspots: {(gdf['hotspot_type'] == 'Coldspot').sum()}")

/opt/anaconda3/envs/conda1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/conda1/lib/python3.13/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


Map saved! Hotspots: 6, Coldspots: 1


In [9]:
m